In [1]:
!pip install pytesseract pillow ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 7.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 17.1 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [pytesseract]
   ---------- ----------------------------- 1/4 [pytesseract]
   ------------------------------ --------- 3/4 [ipywidgets]
   ------------------------------ --------- 3/4 [ipywidgets]
   ------------------------------ --------- 3/4 [ipywidgets]
   ------------------------------ --------- 3/4 [ipywidgets]
   ------------------------------ --------- 3/4 [ipywidgets]
   ------------------------------ --------- 3/4 [ipywidgets]
   ------------------------------ --------- 3/4 [ipywidgets]
   ------------------------------ --------- 3/4 [ipywidgets]
   ---------------------------------------- 4/4 [ipywidgets]



In [2]:
import pytesseract
from PIL import Image
import re
import ipywidgets as widgets
from IPython.display import display

In [3]:
def extract_text(image):
    return pytesseract.image_to_string(image)

In [7]:
def parse_receipt_advanced(text):
    data = {
        "store": None,
        "date": None,
        "invoice_no": None,
        "items": [],
        "total": None,
        "gst": None
    }

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    for i, line in enumerate(lines):
        
        # ---- Store ----
        if "KITCHEN STORE" in line:
            data["store"] = line
        
        # ---- Date ----
        if "date" in line.lower():
            parts = line.split(":")
            if len(parts) > 1:
                data["date"] = parts[-1].strip()
        
        # ---- Invoice ----
        if "invoice no" in line.lower():
            parts = line.split(":")
            if len(parts) > 1:
                data["invoice_no"] = parts[-1].strip()
        
        # ---- Total ----
        if "grand total" in line.lower():
            match = re.search(r'\d+\.?\d*', line)
            if match:
                data["total"] = float(match.group())
        
        # ---- GST ----
        if "total gst" in line.lower():
            match = re.search(r'\d+\.?\d*', line)
            if match:
                data["gst"] = float(match.group())

        # ---- Product line detection ----
        if "kettle" in line.lower():
            data["items"].append({"name": line})

    return data

In [8]:
def parse_receipt_advanced(text):
    data = {
        "store": None,
        "date": None,
        "invoice_no": None,
        "items": [],
        "total": None,
        "gst": None
    }

    lines = [l.strip() for l in text.split("\n") if l.strip()]

    for i, line in enumerate(lines):
        
        # ---- Store ----
        if "KITCHEN STORE" in line:
            data["store"] = line
        
        # ---- Date ----
        if "date" in line.lower():
            parts = line.split(":")
            if len(parts) > 1:
                data["date"] = parts[-1].strip()
        
        # ---- Invoice ----
        if "invoice no" in line.lower():
            parts = line.split(":")
            if len(parts) > 1:
                data["invoice_no"] = parts[-1].strip()
        
        # ---- Total ----
        if "grand total" in line.lower():
            match = re.search(r'\d+\.?\d*', line)
            if match:
                data["total"] = float(match.group())
        
        # ---- GST ----
        if "total gst" in line.lower():
            match = re.search(r'\d+\.?\d*', line)
            if match:
                data["gst"] = float(match.group())

        # ---- Product line detection ----
        if "kettle" in line.lower():
            data["items"].append({"name": line})

    return data

In [9]:
upload = widgets.FileUpload(accept='image/*', multiple=False)
input_box = widgets.Text(placeholder="Ask about the receipt...")
send_button = widgets.Button(description="Ask")
output = widgets.Output()

receipt_data = {"data": None}


def on_upload_change(change):
    if upload.value:
        file_info = list(upload.value.values())[0]
        image = Image.open(file_info['content'])
        
        text = extract_text(image)
        parsed = parse_receipt(text)
        
        receipt_data["data"] = parsed
        
        with output:
            print("Receipt processed. You can now ask questions.\n")


def on_send_clicked(b):
    query = input_box.value
    input_box.value = ""
    
    with output:
        print("You:", query)
        response = receipt_chatbot(query, receipt_data["data"])
        print("Bot:", response)
        print()


upload.observe(on_upload_change, names='value')
send_button.on_click(on_send_clicked)

display(upload, output, input_box, send_button)

FileUpload(value=(), accept='image/*', description='Upload')

Output()

Text(value='', placeholder='Ask about the receipt...')

Button(description='Ask', style=ButtonStyle())

AttributeError: 'tuple' object has no attribute 'values'